# PREPROCESSING SCRIPT AFRICA 

This Notebook represents the integrated preprocessing workflow to prepare the input data for the Continental-scale Groundwater flow model with varibale density flow and salt transport for Africa. It has the following sections: 

1) Assign HK to GEO UNIT "Soil", based on SOILGRIDS Soil Texture + PTF <br>
2) GST (GLOBAL SEDIMENT THICKNESS) XYZ DATA -> RASTER DATA <br>
3) DATA HARMONIZATION <br>
4) Assign HK to GEO UNIT "Consolidated Sediments", based on GLiM <br>
5) Assign HK to GEO UNIT "Unconsolidated Sediments", based on GUM <br>
6) Assign POROSITY to all GEOLOGICAL UNITS (Soil, UnconS, ConsS) <br>
7) ASSIGNING GEO/HYDRAULIC PROPERTIES -> NUMERICAL LAYERS (for > 3 numerical layers) <br>
8) HYDRO RIVERS -> Creating the RIV PACKAGE with CRIV, Stage, Botm elevation

### 1) Assign HK to GEO UNIT "Soil", based on SOILGRIDS Soil Texture + PTF

Here we use the Soilgrids soil texture data to translate it via a pedotransfer function (PTF) into hydraulic conductivity values [m/day]. In this case I chose the Cosby function because it uses the sand fraction, but any better fitting function can be integrated and tested based on the fractions of sand, silt and clay that are calculated in this script.

In [ ]:
'''This script processes the clay and sand content rasters into a weighted average per cell'''

# ========================================= CLAY ================================================
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

# =========================
# USER INPUT
# =========================
input_dir = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\SOILGRIDS\clay"
output_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\clay_weighted_5km_3857.tif"

# model extent (EPSG:3857)
xmin, ymin, xmax, ymax = -2145000, -4440000, 5965000, 4645000

cell_size = 5000  # 5 km

# depth weights (meters)
depths = [
    ("0-5cm", 0.05),
    ("5-15cm", 0.10),
    ("15-30cm", 0.15),
    ("30-60cm", 0.30),
    ("60-100cm", 0.40),
    ("100-200cm", 1.00),
]

# =========================
# CREATE TARGET GRID
# =========================
width = int((xmax - xmin) / cell_size)
height = int((ymax - ymin) / cell_size)

transform = rasterio.transform.from_origin(xmin, ymax, cell_size, cell_size)

# containers
weighted_sum = np.zeros((height, width), dtype=np.float32)
weight_total = np.zeros((height, width), dtype=np.float32)

# =========================
# PROCESS EACH DEPTH
# =========================
for depth_name, thickness in depths:
    print(f"Processing depth: {depth_name}")

    filepath = os.path.join(input_dir, f"clay_5km_{depth_name}.tif")

    with rasterio.open(filepath) as src:

        # prepare destination array
        dest = np.zeros((height, width), dtype=np.float32)

        # reproject + resample to model grid
        reproject(
            source=rasterio.band(src, 1),
            destination=dest,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs="EPSG:3857",
            resampling=Resampling.average
        )

        # mask nodata
        if src.nodata is not None:
            dest = np.where(dest == src.nodata, np.nan, dest)

        # accumulate weighted sum
        weighted_sum += dest * thickness
        weight_total += thickness

# =========================
# FINAL WEIGHTED AVERAGE
# =========================
clay_final = weighted_sum / weight_total

# optional: set NaN to nodata
nodata_value = -9999
clay_final = np.where(np.isnan(clay_final), nodata_value, clay_final)

# =========================
# SAVE OUTPUT
# =========================
with rasterio.open(
    output_file,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=nodata_value,
) as dst:
    dst.write(clay_final, 1)

print("Done. Output saved to:", output_file)




# ======================================= SAND ===============================================
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

# =========================
# USER INPUT
# =========================
input_dir = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\SOILGRIDS\sand"
output_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\sand_weighted_5km_3857.tif"

# model extent (EPSG:3857)
xmin, ymin, xmax, ymax = -2145000, -4440000, 5965000, 4645000

cell_size = 5000  # 5 km

# depth weights (meters)
depths = [
    ("0-5cm", 0.05),
    ("5-15cm", 0.10),
    ("15-30cm", 0.15),
    ("30-60cm", 0.30),
    ("60-100cm", 0.40),
    ("100-200cm", 1.00),
]

# =========================
# CREATE TARGET GRID
# =========================
width = int((xmax - xmin) / cell_size)
height = int((ymax - ymin) / cell_size)

transform = rasterio.transform.from_origin(xmin, ymax, cell_size, cell_size)

# containers
weighted_sum = np.zeros((height, width), dtype=np.float32)
weight_total = np.zeros((height, width), dtype=np.float32)

# =========================
# PROCESS EACH DEPTH
# =========================
for depth_name, thickness in depths:
    print(f"Processing depth: {depth_name}")

    filepath = os.path.join(input_dir, f"sand_5km_{depth_name}.tif")

    with rasterio.open(filepath) as src:

        # prepare destination array
        dest = np.zeros((height, width), dtype=np.float32)

        # reproject + resample to model grid
        reproject(
            source=rasterio.band(src, 1),
            destination=dest,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs="EPSG:3857",
            resampling=Resampling.average
        )

        # mask nodata
        if src.nodata is not None:
            dest = np.where(dest == src.nodata, np.nan, dest)

        # accumulate weighted sum
        weighted_sum += dest * thickness
        weight_total += thickness

# =========================
# FINAL WEIGHTED AVERAGE
# =========================
sand_final = weighted_sum / weight_total

# optional: set NaN to nodata
nodata_value = -9999
sand_final = np.where(np.isnan(sand_final), nodata_value, sand_final)

# =========================
# SAVE OUTPUT
# =========================
with rasterio.open(
    output_file,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=nodata_value,
) as dst:
    dst.write(sand_final, 1)

print("Done. Output saved to:", output_file)

# ================================= COMPUTE SILT CONTENT IN % ====================================
import rasterio
import numpy as np

# =========================
# USER INPUT
# =========================
sand_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\sand_weighted_5km_3857.tif"
clay_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\clay_weighted_5km_3857.tif"

output_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\silt_weighted_5km_3857.tif"

nodata_value = -9999

# =========================
# LOAD INPUT RASTERS
# =========================
with rasterio.open(sand_file) as sand_src, rasterio.open(clay_file) as clay_src:

    # sanity check: grids must match
    assert sand_src.shape == clay_src.shape, "Raster shapes do not match"
    assert sand_src.transform == clay_src.transform, "Transforms do not match"
    assert sand_src.crs == clay_src.crs, "CRS do not match"

    sand = sand_src.read(1).astype(np.float32)
    clay = clay_src.read(1).astype(np.float32)

    profile = sand_src.profile

# =========================
# HANDLE NODATA
# =========================
sand = np.where(sand == sand_src.nodata, np.nan, sand)
clay = np.where(clay == clay_src.nodata, np.nan, clay)

# =========================
# COMPUTE SILT
# =========================
silt = 100.0 - sand - clay

# =========================
# CLEANUP / PHYSICAL CONSTRAINTS
# =========================
# enforce bounds
silt = np.clip(silt, 0, 100)

# where inputs are nan → output nan
mask = np.isnan(sand) | np.isnan(clay)
silt[mask] = np.nan

# =========================
# WRITE OUTPUT
# =========================
silt_out = np.where(np.isnan(silt), nodata_value, silt)

profile.update(dtype="float32", nodata=nodata_value)

with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(silt_out.astype(np.float32), 1)

print("Done. Silt raster saved to:", output_file)




#====================================================================================
#                            ASSIGN HK VIA COSBY FUNCTION
#====================================================================================

import rasterio
import numpy as np


# =========================
# USER INPUT
# =========================
sand_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\sand_weighted_5km_3857.tif"
output_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\hk_soil_cosby_5km.tif"

nodata_value = -9999

# =========================
# LOAD INPUT RASTER
# =========================
with rasterio.open(sand_file) as src:
    sand = src.read(1).astype(np.float32)
    profile = src.profile

# =========================
# HANDLE NODATA
# =========================
sand = np.where(sand == src.nodata, np.nan, sand)

# =========================
# COSBY EQUATION
# Ksat = 0.0070556 * 10^(-0.884 + 0.0153 * sand%)
# Units: cm/s
# =========================
K_cm_per_s = 0.0070556 * (10 ** (-0.884 + 0.0153 * sand))

# =========================
# CONVERT TO m/day
# =========================
# cm/s → m/s → m/day
K_m_per_day = K_cm_per_s * 0.01 * 86400.0

# =========================
# CLEANUP / STABILITY
# =========================
# remove unrealistic extremes
K_m_per_day = np.clip(K_m_per_day, 1e-5, 100)

# apply nodata mask
K_m_per_day[np.isnan(sand)] = np.nan

# =========================
# WRITE OUTPUT
# =========================
K_out = np.where(np.isnan(K_m_per_day), nodata_value, K_m_per_day)

profile.update(dtype="float32", nodata=nodata_value)

with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(K_out.astype(np.float32), 1)

print("Done. Cosby hydraulic conductivity saved to:", output_file)


### 2) GST (GLOBAL SEDIMENT THICKNESS) XYZ DATA -> RASTER DATA

The Global Sediment thickness data from Bird & Mooney is provided in an XYZ format (shapefile) - this script translates it into a continuous raster, later in the global data harmonization it will be further processed.

In [ ]:
'''This script is for converting GST points -> raster and then change CRS, clip to extent and change resolution to 10 km'''

# ==========================================================================
#                 PART 1: SAVE POINTS (SHP) AS RASTER IN EPSG:4326
# ==========================================================================

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from tqdm import tqdm  

# --- INPUTS ---
shapefile = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GST\gst_points_epsg4326_small.shp"   # point layer
value_field = "meter"
res = 0.125

# --- LOAD DATA ---
gdf = gpd.read_file(shapefile)

# Ensure CRS is correct
if gdf.crs.to_string() != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# --- GET EXTENT FROM DATA ---
xmin, ymin, xmax, ymax = gdf.total_bounds

# Snap extent to grid (VERY IMPORTANT)
xmin = np.floor(xmin / res) * res
ymin = np.floor(ymin / res) * res
xmax = np.ceil(xmax / res) * res
ymax = np.ceil(ymax / res) * res

# --- COMPUTE RASTER SIZE ---
width = int(round((xmax - xmin) / res))
height = int(round((ymax - ymin) / res))

print(f"Raster size: {width} x {height}")

# --- EXTRACT DATA ---
xs = gdf.geometry.x.values
ys = gdf.geometry.y.values
vals = gdf[value_field].values

# --- CREATE RASTER ARRAY ---
raster = np.full((height, width), np.nan, dtype=np.float32)

# --- FILL WITH PROGRESS BAR ---
for i in tqdm(range(len(vals)), desc="Rasterizing points"):
    col = int((xs[i] - xmin) / res)
    row = int((ymax - ys[i]) / res)

    if 0 <= row < height and 0 <= col < width:
        raster[row, col] = vals[i]

# --- CHECKS ---
missing = np.isnan(raster).sum()
print(f"Missing cells: {missing}")
print(f"Min value: {np.nanmin(raster)}")
print(f"Max value: {np.nanmax(raster)}")

# --- WRITE GEOTIFF ---
transform = from_origin(xmin, ymax, res, res)

output_file = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GST\GST_python_epsg4326.tif"

with rasterio.open(
    output_file,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype=raster.dtype,
    crs="EPSG:4326",
    transform=transform,
    nodata=-9999
) as dst:
    out = np.where(np.isnan(raster), -9999, raster)
    dst.write(out, 1)

print("Raster written:", output_file)


### 3) DATA HARMONIZATION

This is the longest section, combined of many individual preprocessing scripts, that all include either rasterization, resampling, reprojection, no data handling, unit conversion, clipping, etc. 

I wanted to combine those harmonization steps of most data sets, although some data has to be handled outside this script, since it builds upon the data processed in the harmonization here. To preserve the right order of processing scripts, only the following data was included into the harmonization: 

GEBCO,<br>
Soil thickness,<br>
Recharge,<br>
GST,<br>
Unconsolidated sediment thickness,<br>
Soil hydraulic conductivity (HK),<br>
Aridity index,<br>
Domain and inland masks,<br>
Lakes,<br>
GLHYMPS (Porosity)

In [ ]:
# ============================================================
# Harmonizes raster and vector inputs to a shared 10 km grid.
# ============================================================


import os
import numpy as np
import geopandas as gpd
import xarray as xr
import rasterio

from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from rasterio.features import rasterize, geometry_mask



# RASTERS
GEBCO_PATH = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\unharmonized\Africa_GEBCO.tif" # GEBCO data *
SOIL_PATH = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\unharmonized\Africa_SOIL.tif"   # Soil depth *
RECHARGE_PATH = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\Recharge\Pazola(2024)\RF_RK_recharge_01.tif" # Recharge data
ARIDITY_PATH = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\Aridity_index\ai_v31_yr.tif"
UNCONSOLIDATED_PATH = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\_WORLD_SU_thk_est_mean_bestfit_thk_mWORLD_BESTFIT.tif"
GST_PATH = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GST\GST_python_epsg4326.tif"
HK_SOIL_PATH = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS\hk_soil_cosby_5km.tif"

# * (here: data was already clipped to smaller extend manually to speed up processing, but should work with global data too)


# VECTORS
DOMAIN_SHP = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\africa_domain\Africa_domain_epsg3857.shp"
INLAND_SHP = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\africa_inland\Africa_inland.shp"
LAKES_SHP = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\HydroLAKES\HydroLAKES_polys_v10_shp\HydroLAKES_polys_v10.shp"
GLHYMPS_SHP = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\GLHYMPS\GLHYMPS.shp"


# OUTPUT
OUT_DIR = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\IMOD\Inputs\10_x_10_km"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 4. SHARED MODEL GRID
# ============================================================

TARGET_CRS = "EPSG:3857"
RES = 10_000  # 10 km

XMIN = -2145000
YMIN = -4440000
XMAX = 5965000
YMAX = 4645000

WIDTH = int((XMAX - XMIN) / RES)
HEIGHT = int((YMAX - YMIN) / RES)

TRANSFORM = from_origin(
    XMIN,
    YMAX,
    RES,
    RES
)

# Common intermediate raster nodata value
RAW_NODATA = -9999.0

# Final continuous raster nodata value
FINAL_NODATA = -999.0

print("Model grid")
print("----------------------------")
print(f"CRS:       {TARGET_CRS}")
print(f"Resolution: {RES} m")
print(f"Width:      {WIDTH} columns")
print(f"Height:     {HEIGHT} rows")
print(f"Pixel size: {RES} x {RES} m")
print(f"Actual X bounds: {XMIN} to {XMIN + WIDTH * RES}")
print(f"Actual Y bounds: {YMAX - HEIGHT * RES} to {YMAX}")


# ============================================================
# 5. COMMON RASTER HARMONIZATION FUNCTION
# ============================================================

def reproject_to_grid(
    input_path,
    output_path,
    name,
    resampling=Resampling.bilinear,
    dst_nodata=RAW_NODATA
):
    """
    Reproject a single-band raster onto the fixed model grid.

    Dataset-specific resampling is selected when calling
    the function. Source nodata values and NaNs are handled
    before reprojection.
    """

    print(f"\nReprojecting: {name}")

    with rasterio.open(input_path) as src:

        if src.crs is None:
            raise ValueError(
                f"{name} has no CRS: {input_path}"
            )

        if src.count != 1:
            raise ValueError(
                f"{name} has {src.count} bands. "
                "This function expects a single-band raster."
            )

        src_data = src.read(1).astype(np.float32)

        # Preserve the source nodata definition.
        src_nodata = src.nodata

        # Ensure NaNs are represented by a nodata value.
        if src_nodata is None:
            src_nodata = RAW_NODATA

        src_data[~np.isfinite(src_data)] = src_nodata

        # Destination grid
        dst_data = np.full(
            (HEIGHT, WIDTH),
            dst_nodata,
            dtype=np.float32
        )

        reproject(
            source=src_data,
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=TRANSFORM,
            dst_crs=TARGET_CRS,
            dst_nodata=dst_nodata,
            resampling=resampling
        )

        # Preserve source metadata where relevant, but enforce
        # the common output grid and GeoTIFF settings.
        profile = src.profile.copy()

        profile.update({
            "driver": "GTiff",
            "height": HEIGHT,
            "width": WIDTH,
            "count": 1,
            "dtype": "float32",
            "crs": TARGET_CRS,
            "transform": TRANSFORM,
            "nodata": dst_nodata,
            "compress": "lzw"
        })

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(dst_data, 1)

    valid = dst_data[
        np.isfinite(dst_data) & (dst_data != dst_nodata)
    ]

    print(f"Saved: {output_path}")
    print(f"Valid cells: {valid.size:,}")

    if valid.size:
        print(f"Min: {valid.min():.4f}")
        print(f"Max: {valid.max():.4f}")

    return output_path


# ============================================================
# 6. COMMON VECTOR FUNCTIONS
# ============================================================

def load_vector(path, name):
    """Load a vector dataset and reproject it to the model CRS."""

    print(f"\nLoading {name}...")

    gdf = gpd.read_file(path)

    if gdf.crs is None:
        raise ValueError(
            f"{name} has no CRS defined: {path}"
        )

    gdf = gdf.to_crs(TARGET_CRS)

    # Remove missing or empty geometries.
    gdf = gdf.loc[
        gdf.geometry.notna() & ~gdf.geometry.is_empty
    ].copy()

    print(f"Features: {len(gdf):,}")

    return gdf


def rasterize_vector(
    gdf,
    name,
    output_path,
    fill=0,
    burn_value=1,
    dtype="uint8",
    nodata=255
):
    """
    Rasterize a vector dataset on the shared model grid.

        1 = inside the geometry
        0 = outside the geometry

    Nodata is kept separate from the valid binary values.
    """

    print(f"\nRasterizing {name}...")

    shapes = [
        (geom, burn_value)
        for geom in gdf.geometry
        if geom is not None and not geom.is_empty
    ]

    arr = rasterize(
        shapes,
        out_shape=(HEIGHT, WIDTH),
        transform=TRANSFORM,
        fill=fill,
        dtype=dtype
    )

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=HEIGHT,
        width=WIDTH,
        count=1,
        dtype=dtype,
        crs=TARGET_CRS,
        transform=TRANSFORM,
        nodata=nodata,
        compress="lzw"
    ) as dst:
        dst.write(arr, 1)

    print(f"Saved: {output_path}")
    print(f"Unique values: {np.unique(arr)}")

    return arr

# ============================================================
# 7. LOAD DOMAIN AND INLAND POLYGONS
# ============================================================

domain = load_vector(DOMAIN_SHP, "Model domain")
inland = load_vector(INLAND_SHP, "Inland polygons")


# ============================================================
# 8. CREATE DOMAIN AND INLAND MASKS
# ============================================================

domain_mask_path = os.path.join(OUT_DIR, "domain_10km.tif")
inland_mask_path = os.path.join(OUT_DIR, "inland_10km.tif")

domain_mask_array = rasterize_vector(
    domain,
    "Domain mask",
    domain_mask_path
)

inland_mask_array = rasterize_vector(
    inland,
    "Inland mask",
    inland_mask_path
)


# ============================================================
# 9. PROCESS LAKES
# Reproject -> intersect with domain -> rasterize
# ============================================================

lakes = load_vector(LAKES_SHP, "Lakes")

print("\nClipping lakes to model domain...")

lakes = gpd.overlay(
    lakes,
    domain[["geometry"]],
    how="intersection"
)

# Remove empty geometries resulting from the intersection.
lakes = lakes.loc[
    lakes.geometry.notna() & ~lakes.geometry.is_empty
].copy()

print(f"Remaining lake polygons: {len(lakes):,}")

lakes_path = os.path.join(OUT_DIR, "lakes_10km.tif")

lake_array = rasterize_vector(
    lakes,
    "Lakes",
    lakes_path
)

print("Lake cells:", int(np.sum(lake_array == 1)))
print("Non-lake cells:", int(np.sum(lake_array == 0)))


# ============================================================
# 10. PROCESS GLHYMPS
# Reproject -> bounding-box crop -> save vector
# ============================================================

glhymps = load_vector(GLHYMPS_SHP, "GLHYMPS")

print("\nCropping GLHYMPS to model grid extent...")

# Preserve the original GLHYMPS workflow: bounding-box crop.
# This is NOT a polygon intersection with the domain boundary.
glhymps_clip = glhymps.cx[
    XMIN:XMIN + WIDTH * RES,
    YMAX - HEIGHT * RES:YMAX
].copy()

print(f"Remaining GLHYMPS features: {len(glhymps_clip):,}")

# GeoPackage avoids the field-name and file-management
# limitations of the Shapefile format.
glhymps_output = os.path.join(
    OUT_DIR,
    "GLHYMPS_Africa_3857.gpkg"
)

if os.path.exists(glhymps_output):
    os.remove(glhymps_output)

glhymps_clip.to_file(
    glhymps_output,
    driver="GPKG"
)

print("Saved:", glhymps_output)
print("CRS:", glhymps_clip.crs)
print("Extent:", glhymps_clip.total_bounds)


# ============================================================
# 11. REPROJECT CONTINUOUS RASTERS
# ============================================================

# Each dataset keeps its own resampling method.
# All outputs use the same dimensions, CRS, and transform.

raw_paths = {}

# GEBCO: bilinear ----------------------------
raw_paths["gebco"] = os.path.join(
    OUT_DIR, "gebco_10km_raw.tif"
)

reproject_to_grid(
    GEBCO_PATH,
    raw_paths["gebco"],
    "GEBCO",
    resampling=Resampling.bilinear
)


# Soil thickness: bilinear ---------------------
raw_paths["soil"] = os.path.join(
    OUT_DIR, "soil_10km_raw.tif"
)

reproject_to_grid(
    SOIL_PATH,
    raw_paths["soil"],
    "Soil thickness",
    resampling=Resampling.bilinear
)


# Recharge: bilinear ----------------------------
raw_paths["recharge"] = os.path.join(
    OUT_DIR, "recharge_10km_raw.tif"
)

reproject_to_grid(
    RECHARGE_PATH,
    raw_paths["recharge"],
    "Recharge",
    resampling=Resampling.bilinear
)


# GST: nearest-neighbour -------------------------
# Replaces the separate GST reprojection script.
raw_paths["gst"] = os.path.join(
    OUT_DIR, "gst_10km_raw.tif"
)

reproject_to_grid(
    GST_PATH,
    raw_paths["gst"],
    "GST",
    resampling=Resampling.nearest
)


# Unconsolidated sediment thickness: bilinear -----
raw_paths["unconsolidated"] = os.path.join(
    OUT_DIR, "unconsolidated_10km_raw.tif"
)

reproject_to_grid(
    UNCONSOLIDATED_PATH,
    raw_paths["unconsolidated"],
    "Unconsolidated sediment thickness",
    resampling=Resampling.bilinear
)


# Soil hydraulic conductivity: --------------------
# average resampling from the 5 km source to the 10 km grid.
# Replaces the separate HK resampling script.
raw_paths["hk_soil"] = os.path.join(
    OUT_DIR, "hk_soil_10km_raw.tif"
)

reproject_to_grid(
    HK_SOIL_PATH,
    raw_paths["hk_soil"],
    "Soil hydraulic conductivity",
    resampling=Resampling.average
)


# Aridity index: bilinear --------------------------
raw_paths["aridity"] = os.path.join(
    OUT_DIR, "aridity_index_10km_raw.tif"
)

reproject_to_grid(
    ARIDITY_PATH,
    raw_paths["aridity"],
    "Aridity index",
    resampling=Resampling.bilinear
)

print("\nAll raster reprojections completed.")


# ============================================================
# 12. LOAD HARMONIZED RASTERS
# ============================================================

def load_raw_array(path):
    """
    Load the single-band harmonized raster.
    Convert declared nodata values to NaN for processing.
    """

    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata

    if nodata is not None:
        if np.isnan(nodata):
            arr[~np.isfinite(arr)] = np.nan
        else:
            arr[arr == nodata] = np.nan

    arr[~np.isfinite(arr)] = np.nan

    return arr


gebco = load_raw_array(raw_paths["gebco"])
soil = load_raw_array(raw_paths["soil"])            # soil depth
recharge = load_raw_array(raw_paths["recharge"])
gst = load_raw_array(raw_paths["gst"])
unconsolidated = load_raw_array(raw_paths["unconsolidated"])
hk_soil = load_raw_array(raw_paths["hk_soil"])
aridity = load_raw_array(raw_paths["aridity"])


# ============================================================
# 13. SOIL UNIT CONVERSION
# cm -> m
# ============================================================

soil = soil / 100.0


# ============================================================
# 14. CLEAN MISSING VALUES
# ============================================================

# GEBCO, soil depth, recharge, GST, and unconsolidated sediment
# missing values are replaced with zero.
#
# HK retains NaN to distinguish missing data from zero HK.

gebco = np.nan_to_num(
    gebco, nan=0.0, posinf=0.0, neginf=0.0
)

soil = np.nan_to_num(
    soil, nan=0.0, posinf=0.0, neginf=0.0
)

recharge = np.nan_to_num(
    recharge, nan=0.0, posinf=0.0, neginf=0.0
)

gst = np.nan_to_num(
    gst, nan=0.0, posinf=0.0, neginf=0.0
)

unconsolidated = np.nan_to_num(
    unconsolidated, nan=0.0, posinf=0.0, neginf=0.0
)

# HK deliberately retains NaN.
hk_soil[~np.isfinite(hk_soil)] = np.nan


# ============================================================
# 15. REMOVE NEGATIVE THICKNESS VALUES
# ============================================================

soil[soil < 0] = 0
unconsolidated[unconsolidated < 0] = 0
gst[gst < 0] = 0


# ============================================================
# 16. APPLY INLAND AND DOMAIN MASKS
# ============================================================

inland_mask_bool = inland_mask_array == 1
domain_mask_bool = domain_mask_array == 1

# Inland-based processing
recharge_filled = np.where(
    inland_mask_bool, recharge, 0.0
)

soil_filled = np.where(
    inland_mask_bool, soil, 0.0
)

# GEBCO remains unmasked
gebco_filled = gebco.copy()

hk_soil_filled = np.where(
    inland_mask_bool, hk_soil, np.nan
)

# Domain-based processing
gst_filled = np.where(
    domain_mask_bool, gst, 0.0
)

unconsolidated_filled = np.where(
    domain_mask_bool, unconsolidated, 0.0
)


# ============================================================
# 17. APPLY SOIL/HK DEPENDENCY
# HK only where soil thickness > 0 m
# ============================================================

soil_filled = np.maximum(soil_filled, 0.0)

hk_soil_filled = np.where(
    soil_filled > 0,
    hk_soil_filled,
    np.nan
)


# ============================================================
# 18. RECHARGE UNIT CONVERSION
# mm/year -> m/day
# ============================================================

recharge_filled = recharge_filled / 1000.0 / 365.0


# ============================================================
# 19. PROCESS ARIDITY INDEX
# Apply domain mask, fill missing values with zero, convert units (divided by 10,000).
# ============================================================

aridity_filled = np.where(
    domain_mask_bool,
    aridity,
    0.0
)

aridity_filled = np.nan_to_num(
    aridity_filled,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

aridity_filled = aridity_filled / 10000.0


# ============================================================
# 20. BASIC ARRAY CHECKS
# ============================================================

arrays = {
    "GEBCO": gebco_filled,
    "Soil": soil_filled,
    "Recharge": recharge_filled,
    "GST": gst_filled,
    "Unconsolidated": unconsolidated_filled,
    "HK soil": hk_soil_filled,
    "Aridity": aridity_filled,
}

for name, arr in arrays.items():
    assert arr.shape == (HEIGHT, WIDTH), (
        f"{name} does not match the target grid: {arr.shape}"
    )

print("All processed rasters match the target grid.")


# ============================================================
# 21. SAVE FINAL CONTINUOUS RASTERS
# ============================================================

def save_clean_raster(data, filename, nodata=FINAL_NODATA):
    """
    Save a single-band float32 raster on the fixed model grid.

    NaNs and infinite values are replaced with the declared
    nodata value before writing.
    """

    output_path = os.path.join(OUT_DIR, filename)

    arr = np.asarray(data, dtype=np.float32).copy()

    arr[~np.isfinite(arr)] = nodata

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=HEIGHT,
        width=WIDTH,
        count=1,
        dtype="float32",
        crs=TARGET_CRS,
        transform=TRANSFORM,
        nodata=nodata,
        compress="lzw"
    ) as dst:
        dst.write(arr, 1)

    print(f"Saved: {output_path}")

    return output_path


# Save final model inputs --------------------
save_clean_raster(
    gebco_filled,
    "gebco_10km.tif"
)

save_clean_raster(
    soil_filled,
    "soil_thickness_10km.tif"
)

save_clean_raster(
    recharge_filled,
    "recharge_10km.tif"
)

save_clean_raster(
    gst_filled,
    "gst_10km.tif"
)

save_clean_raster(
    unconsolidated_filled,
    "unconsolidated_10km.tif"
)

save_clean_raster(
    hk_soil_filled,
    "hk_soil_10km.tif"
)

save_clean_raster(
    aridity_filled,
    "aridity_index_10km.tif"
)


# ============================================================
# 22. VALIDATION STATISTICS
# ============================================================

def print_stats(name, arr, units=""):
    """Print statistics for valid, finite cells."""

    valid = np.asarray(arr)
    valid = valid[np.isfinite(valid)]

    print(f"\n{name} {f'({units})' if units else ''}")
    print("-" * 40)

    if valid.size == 0:
        print("No valid cells.")
        return

    print(f"Valid cells: {valid.size:,}")
    print(f"Min:         {valid.min():.6g}")
    print(f"Max:         {valid.max():.6g}")
    print(f"Mean:        {valid.mean():.6g}")


print_stats("Recharge", recharge_filled, "m/day")
print_stats("Soil thickness", soil_filled, "m")
print_stats(
    "Unconsolidated sediment thickness",
    unconsolidated_filled,
    "m"
)
print_stats("GST", gst_filled, "m")
print_stats("Aridity index", aridity_filled)

print(
    "\nCells with zero soil but positive HK:",
    int(
        np.sum(
            (soil_filled == 0)
            & np.isfinite(hk_soil_filled)
            & (hk_soil_filled > 0)
        )
    )
)

print(
    "HK cells with missing values:",
    int(np.isnan(hk_soil_filled).sum())
)


# ============================================================
# 23. CHECK ALL SAVED RASTERS
# ============================================================

raster_outputs = [
    "gebco_10km.tif",
    "soil_thickness_10km.tif",
    "recharge_10km.tif",
    "gst_10km.tif",
    "unconsolidated_10km.tif",
    "hk_soil_10km.tif",
    "aridity_index_10km.tif",
    "domain_10km.tif",
    "inland_10km.tif",
    "lakes_10km.tif",
]

print("\nFINAL RASTER VALIDATION")
print("=" * 60)

for filename in raster_outputs:

    path = os.path.join(OUT_DIR, filename)

    with rasterio.open(path) as src:

        print(f"\n{filename}")
        print(f"  CRS:       {src.crs}")
        print(f"  Size:      {src.width} x {src.height}")
        print(f"  Transform: {src.transform}")
        print(f"  Nodata:    {src.nodata}")

        assert src.crs.to_string() == TARGET_CRS
        assert src.width == WIDTH
        assert src.height == HEIGHT
        assert src.transform == TRANSFORM

print("\nAll output rasters passed the grid consistency checks.")
print("Harmonization complete!")
print("Output directory:", OUT_DIR)

### 4) Assign HK to GEO UNIT "Consolidated Sediments", based on GLiM

In [ ]:
''' This script is built to assign HK values to different geology based on GLIM classification and convert the shapefile into a raster that is harmonized'''

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from shapely.geometry import box


# INPUT
input_shp = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLIM\glim_epsg3857_clipped.shp"
output_raster = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\IMOD\Inputs\5_x_5_km\hk_consolidated_5km_xxyy.tif"

attribute = "Litho"  # <-- use full code column

# extent
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000    # original 5 km


# LOOKUP TABLE (HK in m/day)
hk_lookup = {
    "ev__": 0.01,      # evaporites (Domenico, 1990)
    "mt__": 0.01,      # metamorphites (Freeze)
    "mtam": 0.01,
    "mtgr": 0.01,
    "mtpu": 0.01,
    "mtpy": 0.01,
    "pa__": 0.01,      # acidic plutonic (Heath)
    "pb__": 0.01,      # basic plutonic (Heath)
    "pbam": 0.01,
    "pi__": 0.01,      # intermediate plutonic (Heath)
    "pipy": 0.01, 
    "scam": 0.1,       # carbonate sedimentary (Heath), mafic-metamorphic mentioned
    "scmx": 0.5,              # mixed grained
    "scpu": 0.2,              # pure carbonate
    "scsh": 0.1,              # fine grained
    "scss": 1,                # coarse grained
    "smmx": 0.5,       # mixed sedimentary (Heath)
    "smpy": 0.1,              # pyroclastics mentioned
    "smsh": 0.1,              # fine grained
    "smss": 1,                # coarse
    "ssmx": 0.5,       # siliciclastic sedimentary (Heath) mixed
    "sspy": 0.1,             # pyroclastics mentioned
    "sssh": 0.1,             # fine
    "ssss": 1,               # coarse
    "va__": 0.01,   # acidic volcanic 
    "vapy": 0.01,
    "vb__": 0.01,   # basic volcanic
    "vbpu": 0.01,
    "vbpy": 0.01,
    "vi__": 0.01,   # intermediate volcanic
    "viam": 0.01,
    "vipy": 0.01,
}
bedrock_k = 0.01
default_k = 0.01


# =========================
# LOAD + REPROJECT
# =========================
gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# =========================
# CLIP
# =========================
bbox = gpd.GeoDataFrame(
    geometry=[box(xmin, ymin, xmax, ymax)],
    crs="EPSG:3857"
)

gdf = gpd.clip(gdf, bbox)

# =========================
# EXTRACT xxyy FROM xxyyzz
# =========================
gdf["Litho_xxyy"] = (
    gdf[attribute]
    .astype(str)
    .str.lower()
    .str[:4]   # <-- to only get xxyy from the classification
)

# =========================
# CLEAN CLASSES
# =========================

'''
# -> treat water bodies, nd and unconsolidated sediments like hardrock because below them is bedrock
# remove water (based on first 2 letters still works)
gdf = gdf[~gdf["Litho_xxyy"].str[:2].isin(["wb"])]

# remove unconsolidated sediments (all su** classes)
gdf = gdf[~gdf["Litho_xxyy"].str.startswith("su")]

# remove no-data
gdf = gdf[~gdf["Litho_xxyy"].str[:2].isin(["nd"])]'''

# =========================
# ASSIGN HK
# =========================
def map_k(val):
    # unconsolidated sediments → assign default bedrock HK
    if val.startswith("su") or val.startswith("wb") or val.startswith("nd"):
        return bedrock_k
    return hk_lookup.get(val, default_k)

gdf["HK"] = gdf["Litho_xxyy"].apply(map_k)

# =========================
# GRID
# =========================
width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)
transform = from_origin(xmin, ymax, res, res)

# =========================
# RASTERIZE
# =========================
shapes = [(geom, val) for geom, val in zip(gdf.geometry, gdf["HK"])]

hk_raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=-9999,
    dtype="float32"
)

# =========================
# SAVE
# =========================
with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:
    dst.write(hk_raster, 1)

# =========================
# STATS
# =========================
hk_valid = hk_raster[hk_raster > 0]

print("HK min:", float(hk_valid.min()))
print("HK max:", float(hk_valid.max()))
print("Remaining gaps:", np.sum(hk_raster == -9999))



### 5) Assign HK to GEO UNIT "Unconsolidated Sediments", based on GUM

In [ ]:
''' This script is built to assign HK values to unconsolidated sediments by GUM and GLIM classification and convert the shapefile into a raster that is harmonized'''

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
import os
from shapely.geometry import box

# =========================
# INPUT
# =========================
input_shp = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GUM\GUM_epsg3857_clipped.shp"
out_raster = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GUM\hk_unconsolidated_5km_GUM.tif"

# attribute field with codes like "Au", "Cu", ...
attribute = "XX"                                     # <-- column name

# extent (EPSG:3857)
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000  # original 5 km

# =========================
# LOOKUP TABLE 
# =========================
hk_lookup = {
    "Af": 1,      # alluvial fan                depends on position, from gravel to clay       
    "Al": 0.5,    # alluvial-lacustrine depos   
    "At": 0.5,    # terrace deposits            < 30m/d
    "Au": 1,      # undifferentiated alluvial   0.8-800 m/d
    "Cu": 1,      # colluvial
    "Ed": 10,      # dunes                       Coastal dunes 25 m/d, desert dunes 9.3 m/d,
    "El": 1,       # loess                       0.0001-1.7 m/d
    "Er": 1,       # loess derivates
    "Eu": 0.5,    # aeolian
    "Iy": 0.5,    # pyroclasts                  0.0008-8 m/d
    "Lu": 0.5,    # lacustrine                  0.0008-8 m/d
    "Mu": 0.5,    # marine                      0.01-8 m/d
    "Op": 0.5,    # peat                        0.001-13 m/d
    "Ou": 0.5,    # organic
    "Pg": 0.1,     # gypsum
    "Ps": 0.1,     # salt
    "Pu": 0.1,     # evaporites
    "Us": 10,      # sediments undiff
    "Wl": 0,      # water 
    "Wu": 0,      # water
    "Yu": 10,      # coastal dunes
    "Zu": 1       # anthropogenic
}

default_k = -999  # fallback if class missing

# =========================
# LOAD + REPROJECT
# =========================
gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# =========================
# CLIP TO EXTENT
# =========================
bbox = gpd.GeoDataFrame(
    geometry=[box(xmin, ymin, xmax, ymax)],
    crs="EPSG:3857"
)

gdf = gpd.clip(gdf, bbox)

# remove water
gdf = gdf[~gdf[attribute].isin(["Wl", "Wu"])]

# =========================
# CHECK MISSING CLASSES
# =========================
unique_classes = set(gdf[attribute].unique())
defined_classes = set(hk_lookup.keys())

missing_classes = unique_classes - defined_classes

print("MISSING classes:", missing_classes)

# =========================
# ASSIGN HK TO EACH POLYGON
# =========================
def map_k(val):
    return hk_lookup.get(val, default_k)

gdf["HK"] = gdf[attribute].apply(map_k)

# =========================
# CREATE GRID
# =========================
width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)

transform = from_origin(xmin, ymax, res, res)

# =========================
# RASTERIZE (DIRECTLY HK)
# =========================
shapes = [(geom, value) for geom, value in zip(gdf.geometry, gdf["HK"])]

hk_raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0.0,
    dtype="float32"
)

# =========================
# PRINT STATS
# =========================
hk_valid = hk_raster[hk_raster > 0]  # ignore background (0)

if hk_valid.size > 0:
    print("HK min (m/day):", float(hk_valid.min()))
    print("HK max (m/day):", float(hk_valid.max()))
else:
    print("No valid HK values found.")
print("HK min (including zeros):", float(np.min(hk_raster)))

# =========================
# SAVE RASTER
# =========================
with rasterio.open(
    out_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:
    dst.write(hk_raster, 1)

print("HK raster saved:", out_raster)


# =================================================================================================================
#                    Rasterize Unconsolidated Sedis from GliM as backup for missing GUM areas
# =================================================================================================================

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from shapely.geometry import box

# =========================
# INPUT
# =========================
input_shp = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLIM\glim_epsg3857_clipped.shp"
output_raster = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLIM\hk_uncon_5km_GLIM.tif"

attribute = "Litho"

# extent (same as your model grid)
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 10000   # original 5 km

# =========================
# HK LOOKUP (UNCONSOLIDATED ONLY)
# =========================
hk_lookup = {
    "suad": 0.5, #alluvial dep 
    "suds": 10,  #dunes
    "sumx": 1,   #mixed
    "suor": 0.5, #organic
    "supu": 0.1, #pure carbonate
    "sush": 0.5,
    "suss": 1,
}

default_k = 0.001  # important → keeps gaps clean

# =========================
# LOAD
# =========================
gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# =========================
# CLIP TO MODEL DOMAIN
# =========================
bbox = gpd.GeoDataFrame(
    geometry=[box(xmin, ymin, xmax, ymax)],
    crs="EPSG:3857"
)

gdf = gpd.clip(gdf, bbox)

# =========================
# CLEAN + EXTRACT UNCONSOLIDATED
# =========================
gdf = gdf[gdf[attribute].notna()]

# normalize text
gdf[attribute] = gdf[attribute].astype(str).str.lower()

# extract xxyy (first 4 letters)
gdf["Litho_xxyy"] = gdf[attribute].str[:4]

# keep ONLY unconsolidated (su**)
gdf = gdf[gdf["Litho_xxyy"].str.startswith("su")]

# =========================
# ASSIGN HK
# =========================
def map_k(val):
    return hk_lookup.get(val, default_k)

gdf["HK"] = gdf["Litho_xxyy"].apply(map_k)

# remove anything without HK
gdf = gdf[~gdf["HK"].isna()]

# =========================
# GRID
# =========================
width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)
transform = from_origin(xmin, ymax, res, res)

# =========================
# RASTERIZE
# =========================
shapes = [(geom, val) for geom, val in zip(gdf.geometry, gdf["HK"])]

hk_raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    dtype="float32"
)

# =========================
# SAVE
# =========================
with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:
    dst.write(hk_raster, 1)

# =========================
# STATS
# =========================
hk_valid = hk_raster[~np.isnan(hk_raster)]

print("HK min:", float(hk_valid.min()))
print("HK max:", float(hk_valid.max()))
print("Cells filled:", hk_valid.size)

# ===============================================================================================================
#                                 Merge Glim and GUM results for Unconsolidated Sediments
# ===============================================================================================================

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

gum_path  = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GUM\hk_unconsolidated_5km_GUM.tif"
glim_path = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLIM\hk_uncon_5km_GLIM.tif"
consolidated_path = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\IMOD\Inputs\5_x_5_km\hk_consolidated_5km_xxyy.tif"
landmask_path = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\IMOD\Inputs\5_x_5_km\inland_5km.tif" 
out_path  = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\IMOD\Inputs\5_x_5_km\hk_unconsolidated_5km.tif"

# -------------------------
# READ GUM (reference grid)
# -------------------------
with rasterio.open(gum_path) as gum_src:
    gum = gum_src.read(1).astype("float32")
    profile = gum_src.profile
    gum_nodata = gum_src.nodata

# -------------------------
# FUNCTION: normalize to NaN
# -------------------------
def to_nan(arr, nodata):
    out = arr.copy()
    if nodata is not None:
        out[out == nodata] = np.nan
    out[out == 0] = np.nan                   # only if 0 = no data
    return out

gum_nan = to_nan(gum, gum_nodata)

# -------------------------
# READ + ALIGN GLiM
# -------------------------
with rasterio.open(glim_path) as glim_src:
    glim_raw = glim_src.read(1).astype("float32")
    glim_nodata = glim_src.nodata

    glim = np.full_like(gum, np.nan, dtype="float32")

    reproject(
        source=glim_raw,
        destination=glim,
        src_transform=glim_src.transform,
        src_crs=glim_src.crs,
        dst_transform=profile["transform"],
        dst_crs=profile["crs"],
        resampling=Resampling.nearest
    )

glim_nan = to_nan(glim, glim_nodata)

# -------------------------
# MERGE (GUM priority)
# -------------------------
merged = np.where(~np.isnan(gum_nan), gum_nan, glim_nan)

# -------------------------
# READ + ALIGN CONSOLIDATED
# -------------------------
with rasterio.open(consolidated_path) as cons_src:
    cons_raw = cons_src.read(1).astype("float32")
    cons_nodata = cons_src.nodata

    cons = np.full_like(gum, np.nan, dtype="float32")

    reproject(
        source=cons_raw,
        destination=cons,
        src_transform=cons_src.transform,
        src_crs=cons_src.crs,
        dst_transform=profile["transform"],
        dst_crs=profile["crs"],
        resampling=Resampling.nearest
    )

cons_nan = to_nan(cons, cons_nodata)

# -------------------------
# READ + ALIGN LAND MASK
# -------------------------
with rasterio.open(landmask_path) as land_src:
    land_raw = land_src.read(1)

    land = np.full_like(gum, np.nan, dtype="float32")

    reproject(
        source=land_raw,
        destination=land,
        src_transform=land_src.transform,
        src_crs=land_src.crs,
        dst_transform=profile["transform"],
        dst_crs=profile["crs"],
        resampling=Resampling.nearest
    )

# define land (adjust depending on your raster!)
# Option A: land = 1, ocean = 0
land_mask = land == 1

# If instead land is "not nodata", use:
# land_mask = ~np.isnan(land)

# -------------------------
# DEFAULT FILL (STRICT)
# -------------------------
default_k = 0.1

no_uncon = np.isnan(merged)
no_cons  = np.isnan(cons_nan)

# FINAL CONDITION
fill_mask = no_uncon & no_cons & land_mask

merged[fill_mask] = default_k

# -------------------------
# CONVERT NaN -> -9999
# -------------------------
nodata_val = -9999
merged_out = merged.copy()
merged_out[np.isnan(merged_out)] = nodata_val

# -------------------------
# SAVE
# -------------------------
profile.update(dtype="float32", nodata=nodata_val, compress="lzw")

with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(merged_out, 1)

# -------------------------
# STATS
# -------------------------
valid = merged_out != nodata_val

print("Cells (total):", merged_out.size)
print("Cells (valid):", int(valid.sum()))
print("Cells filled with default:", int(np.sum(fill_mask)))
print("HK min:", float(merged_out[valid].min()))
print("HK max:", float(merged_out[valid].max()))

### 6) Assign POROSITY to all GEOLOGICAL UNITS (Soil, UnconS, ConsS)

In [ ]:
# ==========================================================================================
#                             PREPROCESS POROSITY - GEO LAYER 1 (SOIL)
# ==========================================================================================


import rasterio
import numpy as np
import os

# ==========================================================
# PATHS
# ==========================================================

input_dir = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\SOILGRIDS"
out_dir=r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLHYMPS"

sand_file = os.path.join(input_dir, "sand_weighted_5km_3857.tif")
silt_file = os.path.join(input_dir, "silt_weighted_5km_3857.tif")
clay_file = os.path.join(input_dir, "clay_weighted_5km_3857.tif")

output_file = os.path.join(out_dir, "porosity_layer1_5km.tif")

# ==========================================================
# LOAD RASTERS
# ==========================================================

print("Loading SoilGrids fractions...")

with rasterio.open(sand_file) as src:
    sand = src.read(1)
    profile = src.profile.copy()

with rasterio.open(silt_file) as src:
    silt = src.read(1)

with rasterio.open(clay_file) as src:
    clay = src.read(1)

# ==========================================================
# CLEAN DATA
# ==========================================================

sand = np.nan_to_num(sand)
silt = np.nan_to_num(silt)
clay = np.nan_to_num(clay)

# convert percentages if necessary
if sand.max() > 1:

    sand /= 100
    silt /= 100
    clay /= 100

# ==========================================================
# EFFECTIVE POROSITY
# ==========================================================

print("Assigning effective porosity...")

porosity = np.zeros(sand.shape, dtype="float32")

# Sand-dominated
mask = sand >= 0.70 
porosity[mask] = 0.4                  # medium sand 0.16 – 0.46

# Silt-dominated
mask = silt >= 0.50
porosity[mask] = 0.3                  # silt 0.01 – 0.39

# Clay-dominated
mask = clay >= 0.40
porosity[mask] = 0.1                  # clay 0.01 – 0.18

# Mixed soils
mask = porosity == 0
porosity[mask] = 0.25

# ==========================================================
# SAVE
# ==========================================================

profile.update(
    dtype="float32",
    compress="lzw",
    nodata=-9999
)

with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(porosity, 1)

# ==========================================================
# STATISTICS
# ==========================================================

valid = porosity[porosity >= 0]
print("\n=============================")
print("Layer 1 Porosity Statistics")
print("=============================")
print("Min :", float(valid.min()))
print("Max :", float(valid.max()))
print("Mean:", float(valid.mean()))
print("Std :", float(valid.std()))
print()
print("Sand cells :", np.sum(sand >= 0.70))
print("Clay cells :", np.sum(clay >= 0.40))
print("Silt cells :", np.sum(silt >= 0.50))
print("Mixed cells:", np.sum(porosity == 0.25))
print()
print("Saved:")
print(output_file)






#==========================================================================================
#                       PREPROCESS POROSITY - GEO LAYER 2 Uncons Sedis (GLHYMPS)
#==========================================================================================

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from shapely.geometry import box

# ==========================================================
# INPUT
# ==========================================================

input_shp = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLHYMPS\GLHYMPS_Africa_3857.shp"
output_raster = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLHYMPS\porosity_layer2_5km.tif"

# ==========================================================
# MODEL GRID
# ==========================================================

xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000

width = int((xmax-xmin)/res)
height = int((ymax-ymin)/res)

transform = from_origin(xmin, ymax, res, res)

# ==========================================================
# LOAD
# ==========================================================

print("Loading GLHYMPS...")

gdf = gpd.read_file(input_shp)

gdf = gdf.to_crs("EPSG:3857")

# ==========================================================
# FILTER TO MODEL EXTENT (FAST AND ROBUST)
# ==========================================================

print("Filtering to model extent...")

gdf = gdf.cx[xmin:xmax, ymin:ymax]

print(f"Remaining polygons: {len(gdf)}")

# ==========================================================
# CONVERT POROSITY
# ==========================================================

print("Converting porosity...")
# GLHYMPS stores porosity multiplied by 100
gdf["POR"] = gdf["Porosity_x"] / 100.0

# remove impossible values
gdf.loc[gdf["POR"] < 0,"POR"] = np.nan
gdf.loc[gdf["POR"] > 1,"POR"] = np.nan

# Enforce minimum effective porosity
gdf.loc[gdf["POR"] < 0.01, "POR"] = 0.01

# ==========================================================
# RASTERIZE
# ==========================================================

print("Rasterizing...")
shapes = [
    (geom,val)
    for geom,val in zip(gdf.geometry,gdf["POR"])
    if np.isfinite(val)
]

porosity = rasterize(
    shapes,
    out_shape=(height,width),
    transform=transform,
    fill=-9999,
    dtype="float32"
)

# ==========================================================
# SAVE
# ==========================================================

with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:

    dst.write(porosity,1)

# ==========================================================
# STATISTICS
# ==========================================================

valid = porosity[porosity!=-9999]
print()
print("========== Layer 2 ==========")
print(f"Min  : {valid.min():.3f}")
print(f"Max  : {valid.max():.3f}")
print(f"Mean : {valid.mean():.3f}")
print(f"Std  : {valid.std():.3f}")
print(f"Cells: {valid.size}")
print()
print("Saved:")
print(output_raster)






# ==========================================================================================
#                       PREPROCESS POROSITY - GEO LAYER 3 (Consolid Sedis via GLiM)
# ==========================================================================================

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize

# ==========================================================
# INPUT
# ==========================================================

input_shp = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLIM\glim_epsg3857_clipped.shp"
output_raster = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLHYMPS\porosity_layer3_5km.tif"

attribute = "Litho"

# ==========================================================
# MODEL GRID
# ==========================================================

xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000

width = int((xmax-xmin)/res)
height = int((ymax-ymin)/res)

transform = from_origin(xmin, ymax, res, res)

# ==========================================================
# EFFECTIVE POROSITY LOOKUP
# ==========================================================

por_lookup = {

    # evaporites
    "ev__": 0.01,

    # metamorphic
    "mt__": 0.01,
    "mtam": 0.01,
    "mtgr": 0.01,
    "mtpu": 0.01,
    "mtpy": 0.01,

    # plutonic
    "pa__": 0.01,
    "pb__": 0.01,
    "pbam": 0.01,
    "pi__": 0.01,
    "pipy": 0.01,

    # carbonate sedimentary
    "scam": 0.15,
    "scmx": 0.15,
    "scpu": 0.1,
    "scsh": 0.1,
    "scss": 0.2,

    # mixed sedimentary
    "smmx": 0.15,
    "smpy": 0.1,
    "smsh": 0.1,
    "smss": 0.25,

    # siliciclastic sedimentary
    "ssmx": 0.15,
    "sspy": 0.1,
    "sssh": 0.1,
    "ssss": 0.25,

    # volcanic
    "va__": 0.05,
    "vapy": 0.05,
    "vb__": 0.05,
    "vbpu": 0.05,
    "vbpy": 0.05,
    "vi__": 0.05,
    "viam": 0.05,
    "vipy": 0.05,
}

default_porosity = 0.05

# ==========================================================
# LOAD
# ==========================================================

print("Loading GLIM...")

gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# ==========================================================
# FILTER TO MODEL EXTENT
# ==========================================================

print("Filtering...")

gdf = gdf.cx[xmin:xmax, ymin:ymax]

print("Remaining polygons:", len(gdf))

# ==========================================================
# EXTRACT XXYY
# ==========================================================

gdf["Litho_xxyy"] = (
    gdf[attribute]
    .astype(str)
    .str.lower()
    .str[:4]
)

# ==========================================================
# ASSIGN POROSITY
# ==========================================================

def map_porosity(val):

    if val.startswith("su"):
        return 0.25       # unconsolidated sediments

    if val.startswith("wb"):
        return 0.1        # below water

    if val.startswith("nd"):
        return default_porosity

    return por_lookup.get(val, default_porosity)

gdf["POR"] = gdf["Litho_xxyy"].apply(map_porosity)


# ==========================================================
# RASTERIZE
# ==========================================================

print("Rasterizing...")
shapes = [
    (geom,val)
    for geom,val in zip(gdf.geometry,gdf["POR"])
]

porosity = rasterize(
    shapes,
    out_shape=(height,width),
    transform=transform,
    fill=-9999,
    dtype="float32"
)

# ==========================================================
# SAVE
# ==========================================================

with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:

    dst.write(porosity,1)

# ==========================================================
# STATISTICS
# ==========================================================

valid = porosity[porosity != -9999]

print()
print("====================================")
print("Layer 3 Effective Porosity")
print("====================================")
print(f"Minimum : {valid.min():.3f}")
print(f"Maximum : {valid.max():.3f}")
print(f"Mean    : {valid.mean():.3f}")
print(f"Std     : {valid.std():.3f}")
print(f"Cells   : {valid.size}")
print()
print("Saved:")
print(output_raster)





# =====================================================================================
#                     RESAMPLE POROSITY RASTERS from 5 km  -> 10 km
#
# Uses bilinear interpolation.
# =====================================================================================

import os
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import from_origin

# ==========================================================
# INPUT FOLDER
# ==========================================================

input_dir = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\GLHYMPS"

# ==========================================================
# INPUT FILES
# ==========================================================

input_files = [
    "porosity_layer1_5km.tif",
    "porosity_layer2_5km.tif",
    "porosity_layer3_5km.tif"
]

# ==========================================================
# OUTPUT GRID
# ==========================================================

xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 10000

width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)

dst_transform = from_origin(
    xmin,
    ymax,
    res,
    res
)

# ==========================================================
# LOOP
# ==========================================================

for file in input_files:

    print(f"\nProcessing {file}")

    input_path = os.path.join(input_dir, file)
    output_path = os.path.join(
        input_dir,
        file.replace("_5km", "_10km")
    )

    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        profile.update(\
            width=width,
            height=height,
            transform=dst_transform,
            compress="lzw"
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=src.crs,
                resampling=Resampling.bilinear
            )

    print("Saved:", output_path)
print("\nFinished.")


### 7) ASSIGNING GEO/HYDRAULIC PROPERTIES -> NUMERICAL LAYERS (for > 3 numerical layers)

In [ ]:
'''This script assigns the hk, vk and porosity for all numerical model layers (dominant geology per cell)
for 6 numerical layers in this case
'''

# =================================================================================
#                        CASE 3: 6 numerical layers
# =================================================================================

import numpy as np
import os
import rasterio
from rasterio.warp import reproject, Resampling


input_dir = r'C:\GROUNDWATCH\THESIS\Modeling\AFRICA\IMOD\Inputs\10_x_10_km\6_ml'
output_dir = r'C:\GROUNDWATCH\THESIS\Modeling\AFRICA\IMOD\Inputs\10_x_10_km\6_ml'

#Set number of layers here:
nlay = 6


# ------------------------------------------------------------------
# Load mask
# ------------------------------------------------------------------

with rasterio.open(os.path.join(input_dir, 'inland_10km.tif')) as src:
    inland = src.read(1)


# ------------------------------------------------------------------
# Load ORIGINAL geological thicknesses
# These must be the original thicknesses WITHOUT numerical minimums
# ------------------------------------------------------------------

with rasterio.open(os.path.join(input_dir, 'soil_thickness_10km.tif')) as src:
    soil_thk_orig = src.read(1).astype(float)
    profile = src.profile
    # store model grid for transforming seabed litho
    model_transform = src.transform
    model_crs = src.crs
    model_width = src.width
    model_height = src.height

with rasterio.open(os.path.join(input_dir, 'unconsolidated_10km.tif')) as src:
    uncon_thk_orig = src.read(1).astype(float)

with rasterio.open(os.path.join(input_dir, 'gst_10km.tif')) as src:
    gst_thk_orig = src.read(1).astype(float)


# Seabed lithology 
seabed_fp = os.path.join(input_dir, "seabed_litho_epsg3857.tif")

with rasterio.open(seabed_fp) as src:
    seabed_src = src.read(1).astype(np.float32)
    seabed_lithology = np.full(
        (model_height, model_width),
        np.nan,
        dtype=np.float32)
    reproject(
        source=seabed_src,
        destination=seabed_lithology,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src.nodata,
        dst_transform=model_transform,
        dst_crs=model_crs,
        dst_nodata=np.nan,
        resampling=Resampling.nearest
    )

# safety check
print("Model grid shape:", soil_thk_orig.shape)
print("Seabed grid shape:", seabed_lithology.shape)
valid_codes = seabed_lithology[
    np.isfinite(seabed_lithology)
]
print("Seabed lithology codes:", np.unique(valid_codes))

# ------------------------------------------------------------------
# Load ORIGINAL geological HK
# ------------------------------------------------------------------

with rasterio.open(os.path.join(input_dir, 'hk_soil_10km.tif')) as src:
    hk_soil = src.read(1).astype(float)

with rasterio.open(os.path.join(input_dir, 'hk_unconsolidated_10km.tif')) as src:
    hk_uncon = src.read(1).astype(float)

with rasterio.open(os.path.join(input_dir, 'hk_consolidated_10km_xxyy.tif')) as src:
    hk_cons = src.read(1).astype(float)


# ------------------------------------------------------------------
# Load ORIGINAL geological porosity
# ------------------------------------------------------------------

with rasterio.open(os.path.join(input_dir, 'porosity_soil_10km.tif')) as src:
    por_soil = src.read(1).astype(float)

with rasterio.open(os.path.join(input_dir, 'porosity_uncon_10km.tif')) as src:
    por_uncon = src.read(1).astype(float)

with rasterio.open(os.path.join(input_dir, 'porosity_cons_10km.tif')) as src:
    por_cons = src.read(1).astype(float)


# ------------------------------------------------------------------
# Clean original thicknesses
# ------------------------------------------------------------------

soil_thk_orig = np.nan_to_num(soil_thk_orig, nan=0.0)
uncon_thk_orig = np.nan_to_num(uncon_thk_orig, nan=0.0)
gst_thk_orig = np.nan_to_num(gst_thk_orig, nan=0.0)

soil_thk_orig = np.maximum(soil_thk_orig, 0.0)
uncon_thk_orig = np.maximum(uncon_thk_orig, 0.0)

# Consolidated thickness = total sediment thickness
# minus soil and unconsolidated thickness
cons_thk_orig = gst_thk_orig - (
    soil_thk_orig + uncon_thk_orig
)

cons_thk_orig = np.maximum(cons_thk_orig, 0.0)


# ------------------------------------------------------------------
# Calculate geological VK from geological HK
# ------------------------------------------------------------------

vk_soil = hk_soil * 0.1
vk_uncon = hk_uncon * 0.1
vk_cons = hk_cons * 0.1

# -----------------------------------------------------------------
# Numerical 6-layer geometry
#
# L1: soil interval, minimum 10 m
#
# L2 + L3: original unconsolidated interval,
#          minimum total numerical thickness = 20 m
#
# L4 + L5: original consolidated interval,
#          minimum total numerical thickness = 20 m
#
# L6: basal numerical layer below mapped geology.
#     This layer is intended to act as the basal constant-salinity
#     reservoir during transport simulations.

num_thk1 = np.maximum(soil_thk_orig, 10.0)

num_uncon_total = np.maximum(uncon_thk_orig, 20.0)
num_thk2 = num_uncon_total / 2.0
num_thk3 = num_uncon_total / 2.0

num_cons_total = np.maximum(cons_thk_orig, 20.0)
num_thk4 = num_cons_total / 2.0
num_thk5 = num_cons_total / 2.0

# Dedicated basal layer
basal_thickness = 100.0

num_thk6 = np.full_like(
    soil_thk_orig,
    basal_thickness,
    dtype=float
)

num_thickness = np.stack([
    num_thk1,
    num_thk2,
    num_thk3,
    num_thk4,
    num_thk5,
    num_thk6
])


# ------------------------------------------------------------------
# Geological interfaces measured as depth below surface
# ------------------------------------------------------------------

geo_soil_top = np.zeros_like(soil_thk_orig)
geo_soil_bottom = soil_thk_orig

geo_uncon_top = geo_soil_bottom
geo_uncon_bottom = geo_uncon_top + uncon_thk_orig

geo_cons_top = geo_uncon_bottom
geo_cons_bottom = geo_cons_top + cons_thk_orig


# ------------------------------------------------------------------
# Numerical layer interfaces measured as depth below surface
# ------------------------------------------------------------------

num_top = np.zeros_like(num_thickness)
num_bottom = np.zeros_like(num_thickness)

running_depth = np.zeros_like(soil_thk_orig)

for k in range(nlay):

    num_top[k] = running_depth

    num_bottom[k] = (
        running_depth + num_thickness[k]
    )

    running_depth = num_bottom[k]


# ------------------------------------------------------------------
# Function to calculate vertical overlap between:
#
# numerical layer [num_top, num_bottom]
# geological unit [geo_top, geo_bottom]
#
# Returns overlap thickness in metres.
# ------------------------------------------------------------------

def overlap_thickness(num_top, num_bottom, geo_top, geo_bottom):

    overlap = (
        np.minimum(num_bottom, geo_bottom)
        - np.maximum(num_top, geo_top)
    )

    return np.maximum(overlap, 0.0)


# ------------------------------------------------------------------
# Output arrays
# ------------------------------------------------------------------

nrow, ncol = soil_thk_orig.shape

hk_layers = np.zeros((nlay, nrow, ncol), dtype=float)
vk_layers = np.zeros((nlay, nrow, ncol), dtype=float)
por_layers = np.zeros((nlay, nrow, ncol), dtype=float)

# dominant geology:
#
# 0 = undefined / outside model
# 1 = soil
# 2 = unconsolidated
# 3 = consolidated
# 4 = bedrock
# 5 = seabed sediment

# This is useful to save so you can inspect exactly which geological
# material was assigned to every numerical cell.
dominant_geology = np.zeros(
    (nlay, nrow, ncol),
    dtype=np.int8
)

# ------------------------------------------------------------------
# Bedrock default properties
# ------------------------------------------------------------------

hk_bedrock = 0.05
vk_bedrock = hk_bedrock * 0.1
por_bedrock = 0.01   


# ------------------------------------------------------------------
# Assign properties according to DOMINANT GEOLOGICAL MATERIAL
# ------------------------------------------------------------------

for k in range(nlay):

    # Calculate how many metres of each geological material
    # occur inside this numerical layer
    soil_overlap = overlap_thickness(
        num_top[k],
        num_bottom[k],
        geo_soil_top,
        geo_soil_bottom
    )

    uncon_overlap = overlap_thickness(
        num_top[k],
        num_bottom[k],
        geo_uncon_top,
        geo_uncon_bottom
    )

    cons_overlap = overlap_thickness(
        num_top[k],
        num_bottom[k],
        geo_cons_top,
        geo_cons_bottom
    )

    # Stack geological overlaps
    overlaps = np.stack([
        soil_overlap,
        uncon_overlap,
        cons_overlap
    ])

    # Find geology with largest overlap
    dominant = np.argmax(overlaps, axis=0)
    max_overlap = np.max(overlaps, axis=0)

    # Save as 1/2/3 rather than 0/1/2
    dominant_geology[k] = dominant + 1

    # Assign complete material properties according to dominant unit
    mask_soil = dominant == 0
    mask_uncon = dominant == 1
    mask_cons = dominant == 2

    hk_layers[k][mask_soil] = hk_soil[mask_soil]
    vk_layers[k][mask_soil] = vk_soil[mask_soil]
    por_layers[k][mask_soil] = por_soil[mask_soil]

    hk_layers[k][mask_uncon] = hk_uncon[mask_uncon]
    vk_layers[k][mask_uncon] = vk_uncon[mask_uncon]
    por_layers[k][mask_uncon] = por_uncon[mask_uncon]

    hk_layers[k][mask_cons] = hk_cons[mask_cons]
    vk_layers[k][mask_cons] = vk_cons[mask_cons]
    por_layers[k][mask_cons] = por_cons[mask_cons]

    # If this numerical layer lies below all mapped geology,
    # assign bedrock rather than accepting np.argmax -> soil.
    no_overlap = max_overlap <= 0.0

    hk_layers[k][no_overlap] = hk_bedrock
    vk_layers[k][no_overlap] = vk_bedrock
    por_layers[k][no_overlap] = por_bedrock
    dominant_geology[k][no_overlap] = 4

# -----------------------------------------------------------------
# Seabed lithology properties
# -----------------------------------------------------------------
# code : [HK (m/day), porosity (-), VK/HK ratio]
#
# 1  Gravel and coarser
# 2  Sand
# 3  Silt
# 4  Clay
# 5  Calcareous ooze
# 6  Radiolarian ooze
# 7  Diatom ooze
# 8  Sponge spicules
# 9  Mixed calcareous/siliceous ooze
# 10 Shells and coral fragments
# 11 Ash and volcanic sand/gravel
# 12 Siliceous mud
# 13 Fine-grained calcareous sediment

seabed_properties = {
    1:  {"hk": 10.0,  "por": 0.35, "vk_ratio": 0.1},
    2:  {"hk": 5.0,   "por": 0.40, "vk_ratio": 0.1},
    3:  {"hk": 0.50,  "por": 0.50, "vk_ratio": 0.1},
    4:  {"hk": 0.05,  "por": 0.60, "vk_ratio": 0.1},
    5:  {"hk": 0.1,  "por": 0.60, "vk_ratio": 0.1},
    6:  {"hk": 0.1,  "por": 0.70, "vk_ratio": 0.1},
    7:  {"hk": 0.1,  "por": 0.70, "vk_ratio": 0.1},
    8:  {"hk": 0.1,  "por": 0.70, "vk_ratio": 0.1},
    9:  {"hk": 0.1,  "por": 0.60, "vk_ratio": 0.1},
    10: {"hk": 5.0,  "por": 0.40, "vk_ratio": 0.1},
    11: {"hk": 1.0,   "por": 0.40, "vk_ratio": 0.1},
    12: {"hk": 0.05,  "por": 0.60, "vk_ratio": 0.1},
    13: {"hk": 0.05,  "por": 0.60, "vk_ratio": 0.1},
}

hk_seabed = np.full(
    seabed_lithology.shape,
    np.nan,
    dtype=float
)

vk_seabed = np.full(
    seabed_lithology.shape,
    np.nan,
    dtype=float
)

por_seabed = np.full(
    seabed_lithology.shape,
    np.nan,
    dtype=float
)

for code, props in seabed_properties.items():

    mask = seabed_lithology == code
    hk_seabed[mask] = props["hk"]
    vk_seabed[mask] = (props["hk"] * props["vk_ratio"])
    por_seabed[mask] = props["por"]


# ------------------------------------------------------------------
# Offshore section
# ------------------------------------------------------------------

inland_mask = inland == 1
ocean_mask = ~inland_mask

valid_seabed = (
    ocean_mask
    & np.isfinite(seabed_lithology)
    & (seabed_lithology >= 1)
    & (seabed_lithology <= 13)
    & np.isfinite(hk_seabed)
    & np.isfinite(vk_seabed)
    & np.isfinite(por_seabed)
)

# Assign artificial/default properties to the complete offshore column first.
# Valid seabed properties will subsequently overwrite Layers 1 and 2.
for k in range(5):

    hk_layers[k][ocean_mask] = 0.1
    vk_layers[k][ocean_mask] = 0.05
    por_layers[k][ocean_mask] = 0.1

    dominant_geology[k][ocean_mask] = 4

# Layer 6 = basal bedrock everywhere
hk_layers[5][ocean_mask] = hk_bedrock
vk_layers[5][ocean_mask] = vk_bedrock
por_layers[5][ocean_mask] = por_bedrock
dominant_geology[5][ocean_mask] = 4

# Apply seabed lithology properties to offshore Layers 1 and 2.
for k in [0, 1]:
    hk_layers[k][valid_seabed] = hk_seabed[valid_seabed]
    vk_layers[k][valid_seabed] = vk_seabed[valid_seabed]
    por_layers[k][valid_seabed] = por_seabed[valid_seabed]
    dominant_geology[k][valid_seabed] = 5


# New code 5 = seabed sediment
dominant_geology[0][valid_seabed] = 5
dominant_geology[1][valid_seabed] = 5

# ------------------------------------------------------------------
# Save raster function
# ------------------------------------------------------------------

def save_raster(array, filename):

    out_profile = profile.copy()

    out_profile.update(
        dtype=rasterio.float32,
        count=1,
        nodata=np.nan,
        compress='lzw'
    )

    output_fp = os.path.join(
        output_dir,
        filename
    )

    with rasterio.open(
        output_fp,
        'w',
        **out_profile
    ) as dst:

        dst.write(
            array.astype(np.float32),
            1
        )


# ------------------------------------------------------------------
# Save HK, VK and porosity
# ------------------------------------------------------------------

for k in range(nlay):

    layer = k + 1

    save_raster(
        hk_layers[k],
        f"hk_layer{layer}.tif"
    )

    save_raster(
        vk_layers[k],
        f"vk_layer{layer}.tif"
    )

    save_raster(
        por_layers[k],
        f"porosity_layer{layer}.tif"
    )

save_raster(
    hk_seabed,
    "hk_seabed.tif"
)

save_raster(
    vk_seabed,
    "vk_seabed.tif"
)

save_raster(
    por_seabed,
    "porosity_seabed.tif"
)
# ------------------------------------------------------------------
# Save dominant-geology rasters
#
# 0 = outside model
# 1 = soil
# 2 = unconsolidated
# 3 = consolidated
# 4 = bedrock
# 5 = seabed sediment
# ------------------------------------------------------------------

for k in range(nlay):

    layer = k + 1

    save_raster(
        dominant_geology[k],
        f"dominant_geology_layer{layer}.tif"
    )


print("Finished dominant-geology preprocessing.")
print()
print("Dominant geology codes:")
print("0 = undefined / outside model")
print("1 = soil")
print("2 = unconsolidated")
print("3 = consolidated")
print("4 = bedrock")
print("5 = seabed sediment")

### 8) HYDRO RIVERS -> Creating the RIV PACKAGE with CRIV, Stage, Botm elevation

In [ ]:
''' This script preprocesses HydroRIVERS and creates MODFLOW RIV input using the approach of 
calculating geometric river probperties per reach and then summing or averaging them per cell (c_geom), 
also a climate dependent CRIV is calculated (c_climate) and the higher CRIV is chosen '''

import sys
print(sys.executable)
import rasterio
print(rasterio.__file__)

import geopandas as gpd
import numpy as np
from shapely.geometry import box
import rasterio
import pandas as pd
import matplotlib.pyplot as plt
import os
from rasterstats import zonal_stats
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

# In
river_fp = r"C:\GROUNDWATCH\THESIS\Data\Continent Africa\HydroRIVERS\HydroRIVERS_v10_af_shp\HydroRIVERS_v10_af.shp"
dem_fp = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\Inputs\10_x_10_km\gebco_10km.tif"
dem_1km_fp = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\unharmonized\Africa_GEBCO_3857.tif"
soil_fp = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\Inputs\10_x_10_km\soil_thickness_10km.tif"
domain_fp = r"C:\GROUNDWATCH\THESIS\Data\1_QGIS_Processed\africa_domain\Africa_domain.shp"
vk_fp = r'C:\GROUNDWATCH\THESIS\Modeling\AFRICA\Inputs\10_x_10_km\vk_layer1.tif'           # here use vk_layer1 from preprocessing step 10
inland_fp= r'C:\GROUNDWATCH\THESIS\Modeling\AFRICA\Inputs\10_x_10_km\inland_10km.tif'
aridity_fp = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\Inputs\10_x_10_km\aridity_index_clipped_3857.tif"


# Out
riv_package_fp = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\Inputs\10_x_10_km\riv_package_10x10_c5.0.txt"
plot_dir = r"C:\GROUNDWATCH\THESIS\Modeling\AFRICA\Outputs\10_x_10_km\river_package"
os.makedirs(plot_dir, exist_ok=True)
print("Loading paths done")

# domain
xmin, ymin, xmax, ymax = -2145000, -4440000, 5965000, 4645000
cell_size = 10000

# LOAD + REPROJECT + CLIP TO DOMAIN 
gdf = gpd.read_file(river_fp).to_crs(epsg=3857)

domain = gpd.read_file(domain_fp).to_crs(epsg=3857)
gdf = gpd.clip(gdf, domain)                          # clip rivers to domain

# Use all rivers
gdf = gdf[gdf["DIS_AV_CMS"] > 0]

print("Load, reproject, clip done")

# ----------------------------- CREATE GRID -------------------------------------
cols = np.arange(xmin, xmax, cell_size)
rows = np.arange(ymin, ymax, cell_size)

grid_cells = [box(x, y, x+cell_size, y+cell_size) for x in cols for y in rows]
grid = gpd.GeoDataFrame(geometry=grid_cells, crs="EPSG:3857")
grid["cell_id"] = np.arange(len(grid))

grid = gpd.overlay(grid, domain, how="intersection")      # mask the grid with domain

# measure length of river per cell 
intersections = gpd.overlay(gdf, grid, how="intersection")
intersections["length_m"] = intersections.geometry.length

# -------------------------------- LOAD rasters ---------------------------------
with rasterio.open(dem_fp) as src:
    top_arr = src.read(1)
    transform = src.transform

with rasterio.open(soil_fp) as src:
    soil_arr = src.read(1)

with rasterio.open(vk_fp) as src:
    vk_arr = src.read(1)

with rasterio.open(inland_fp) as src:
    inland_arr = src.read(1)

with rasterio.open(aridity_fp) as src:
    aridity_arr = src.read(1)

# apply minimum thickness = 10 m (same as MODFLOW model)
soil_arr = np.nan_to_num(soil_arr, nan=0.0)
soil_arr[soil_arr < 10] = 10.0

nrow, ncol = top_arr.shape

print("load rasters done")

# ----------------  SAMPLE TOP, THICKNESS, VK PER INTERSECTIONS ------------------------

def sample_raster(x, y, arr, transform):
    col, row = ~transform * (x, y)
    col, row = int(col), int(row)
    if 0 <= row < arr.shape[0] and 0 <= col < arr.shape[1]:
        return arr[row, col]
    else:
        return np.nan

intersections["top"] = [
    sample_raster(geom.centroid.x,
                  geom.centroid.y,
                  top_arr,
                  transform)
    for geom in intersections.geometry
]

intersections["vk_layer1"] = [
    sample_raster(geom.centroid.x,
                  geom.centroid.y,
                  vk_arr,
                  transform)
    for geom in intersections.geometry
]

# sample inland mask
intersections["inland"] = [

    sample_raster(
        geom.centroid.x,
        geom.centroid.y,
        inland_arr,
        transform
    )
    for geom in intersections.geometry
]

grid["soil_thickness"] = [
    sample_raster(geom.centroid.x, geom.centroid.y, soil_arr, transform)
    for geom in grid.geometry
]

grid["top"] = [
    sample_raster(
        geom.centroid.x,
        geom.centroid.y,
        top_arr,
        transform
    )
    for geom in grid.geometry
]

grid["aridity"] = [
    sample_raster(
        geom.centroid.x,
        geom.centroid.y,
        aridity_arr,
        transform
    )
    for geom in grid.geometry
]

print("Sampling done")


# REMOVE RIVERS OUTSIDE INLAND 
before_count = len(intersections)
# keep only inland river segments
intersections = intersections[
    intersections["inland"] == 1
].copy()
after_count = len(intersections)
removed = before_count - after_count
print("\nOFFSHORE RIVER FILTER")
print(f"River segments before filter: {before_count}")
print(f"River segments after filter:  {after_count}")
print(f"Removed offshore segments:    {removed}")
print("removing rivers outside inland done")

# Use aridity index for fractions for COND
def river_fraction_from_aridity(ai):
    if np.isnan(ai):    # NaN
        return 0.01
    
    if ai > 6500:       # humid
        return 0.15
    elif ai > 5000:     # subhumid
        return 0.10
    elif ai > 2000:     # semi-arid
        return 0.05
    elif ai > 300:      # arid
        return 0.02
    else:               # hyperarid
        return 0.005

# Define C values for CRIV ≈ Area_surface_water / c
def resistance_from_aridity(ai):
    if np.isnan(ai):     # NaN
        return 20
    if ai > 6500:        # humid
        return 10
    elif ai > 5000:      # subhumid
        return 15
    elif ai > 2000:      # semi-arid
        return 20
    elif ai > 300:       # arid
        return 40
    else:                # hyperarid
        return 100

# ----------------------- COMPUTATIONS PER RIVER SEGMENT-----------------------------------------
 
#  bottom, thickness [m]
grid["bottom"] = grid["top"] - grid["soil_thickness"]
grid["thickness"] = grid["soil_thickness"]

# elevation class 
def elev_class(z):
    if z > 1000:
        return "mountain"
    elif z > 300:
        return "transition"
    else:
        return "lowland"

intersections["elev_class"] = intersections["top"].apply(elev_class)


# width, depth (from q, terrain classification)
def width_depth_factors(elev_class):
    if elev_class == "mountain":
        width_factor = 0.4      # narrower
        depth_factor = 2.5      # deeper
    elif elev_class == "transition":
        width_factor = 0.7
        depth_factor = 1.5
    else:  # lowlands
        width_factor = 1.0      # wider
        depth_factor = 1.0      # shallower
    return width_factor, depth_factor
widths = []
depths = []

for _, row in intersections.iterrows():

    Q = row["DIS_AV_CMS"]

    wf, df = width_depth_factors(row["elev_class"])

    W = (7.2 * (Q ** 0.5)) * wf
    D = (0.27 * (Q ** 0.4)) * df

    W = np.clip(W, 1, 700) 
    D = np.clip(D, 0.5, 50)

    widths.append(W)
    depths.append(D)

intersections["width"] = widths
intersections["depth"] = depths

print(intersections["width"].describe())
print(intersections["depth"].describe())

# riverbed thickness      
# grid["thickness_rb"] = np.clip(0.3 * grid["depth"], 1.0, 20.0)  #old approach (M = 0.3 * depth)
def assign_M(elev_class):
    if elev_class == "mountain":
        return 0.1
    elif elev_class == "transition":
        return 10
    else:             # lowlands
        return 50

intersections["thickness_rb"] = (
    intersections["elev_class"]
    .apply(assign_M)
)

# K_rb conceptual (after terrain) without q class
def assign_K(elev_class): # K_rb value in [m/d]
    if elev_class == "mountain":
        return 0.1
    elif elev_class == "transition":
        return 2
    else:                 # lowlands
        return 5

intersections["Krb_conceptual"] = intersections.apply(
    lambda r: assign_K(
        r["elev_class"]
    ),
    axis=1
)

intersections["Krb"] = np.minimum(
    intersections["Krb_conceptual"],
    intersections["vk_layer1"]
)
print("K_rb done")
# ------------------ AGGREGATE TO CELL GRID ------------------

agg = intersections.groupby("cell_id").agg({
    "length_m": "max",                           #use max instead of sum!
    #"width": "sum",
    "depth": "mean",
    "Krb": "mean",
    "thickness_rb": "mean"
}).reset_index()

# calculate effective width using root-sum-square
width_eff = (
    intersections
    .groupby("cell_id")["width"]
    .apply(lambda x: np.sqrt(np.sum(x**2)))
    .reset_index(name="width")
)
agg = agg.merge(width_eff, on="cell_id")

grid = grid.merge(
    agg,
    on="cell_id",
    how="left"
)

# For computation of C_climate 
cell_area = 10000 * 10000

grid["river_fraction"] = (
    grid["aridity"]
    .apply(river_fraction_from_aridity)
)
grid["c_climate"] = (
    grid["aridity"]
    .apply(resistance_from_aridity)
)
# Compute C_climate
grid["C_climate"] = (
    cell_area
    * grid["river_fraction"]
    / grid["c_climate"]
)

# constrain length at 10km for dense river networks for aggregation per cell
grid["length_m"] = np.minimum(
    grid["length_m"],
    10000
)

print("Aggregation done")


# ----------------------- COMPUTE STAGE AND BED ELEVATION -----------------------
# stage [m] 
# -> 5th percentile of lowest elevation in cell or top-thk/2
stats = zonal_stats(
    grid,
    dem_1km_fp,
    stats=None,
    add_stats={"p5": lambda x: np.nanpercentile(np.asarray(x).astype(float).copy(), 5 )},
    nodata=-9999
)
grid["stage"] = [s["p5"] for s in stats]
# constraints in top and bottom:
grid["stage"] = np.minimum( grid["stage"], grid["top"] - 0.1)   # max DEM - 0.1 m
grid["stage"] = np.maximum( grid["stage"], grid["top"] - 50)    # min DEM - 50

# bed elevation
grid["bed_elev"] = grid["stage"] - grid["depth"]
grid["bed_elev"] = np.minimum(grid["bed_elev"],  grid["stage"] - 0.1) # ensure bed minimum below -50.1m

print("bed elevation done")
      
# --------------------- Calculate CRIV conductance [m²/day] --------------
grid["C_geom"] = (
    grid["Krb"]
    * grid["length_m"]
    * grid["width"]
) / grid["thickness_rb"]
print(grid["C_geom"].describe())

print("computation C_geom done")

# mask river cells
grid["has_river"] = ~grid["C_geom"].isna()
grid["conductance"] = np.nan
river_cells = grid["has_river"]

grid.loc[river_cells, "conductance"] = np.maximum(
    grid.loc[river_cells, "C_geom"],
    grid.loc[river_cells, "C_climate"]
)

print("Cells with top < -50 and river:")
print(
    np.sum(
        (grid["conductance"].notna()) &
        (grid["top"] < -50)
    )
)

# diagnostics
diag_fp = os.path.join(
    plot_dir,
    "river_conductance_diagnostics.csv"
)
grid[
[
    "cell_id",
    "aridity",
    "river_fraction",
    "c_climate",
    "C_climate",
    "C_geom",
    "conductance"
]
].to_csv(
    diag_fp,
    index=False
)
print("Saved:", diag_fp)

# summary stats
summary_fp = os.path.join(
    plot_dir,
    "river_conductance_summary.csv"
)
grid[
[
    "aridity",
    "river_fraction",
    "c_climate",
    "C_climate",
    "C_geom",
    "conductance"
]
].describe().to_csv(
    summary_fp
)
print("Saved:", summary_fp)

# investigate high conductance areas
highC = grid.nlargest(20, "conductance")
print(
    highC[
        [
            "conductance",
            "length_m",
            "width",
            "Krb",
            "thickness_rb",
            "top",
        ]
    ]
)
print("Finished computation. RIV package will be written next.")

# Remove RIV from Ocean
grid["inland"] = [
    sample_raster(
        geom.centroid.x,
        geom.centroid.y,
        inland_arr,
        transform
    )
    for geom in grid.geometry
]
river_before = grid["conductance"].notna().sum()

grid.loc[
    grid["inland"] != 1,
    "conductance"
] = np.nan

river_after = grid["conductance"].notna().sum()

print("Removed river cells:", river_before-river_after)

# ----------------------------- WRITE RIV PACKAGE --------------------------
with open(riv_package_fp, "w") as f:
    for _, row in grid.dropna(subset=["conductance"]).iterrows():

        layer = 1
        r = int((ymax - row.geometry.centroid.y) / cell_size)
        c = int((row.geometry.centroid.x - xmin) / cell_size)

        f.write(f"{layer} {r} {c} "
                f"{row['stage']} {row['conductance']} {row['bed_elev']}\n")

print("write package done")